# 07 — Pairwise Preference Data

## Objective
Inspect chosen and rejected responses, test basic integrity, and explain their supervision signal.

## Background
DPO (Direct Preference Optimization) and reward modeling learn from comparisons. The Med-REFL-DPO dataset automatically constructs medical reasoning preference examples. Such labels are not interchangeable with independently collected human judgments.

### Inspect one preference pair

**Procedure.** Create a clearly illustrative prompt with a preferred and rejected answer and validate basic field integrity.

**Reading the result.** The printed pair shows the data shape required for preference optimization, not expert-annotated results.

In [1]:
# A small invented preference pair tests the field contract; it is not source data.
example = {
    "prompt": "Does association establish causation?",
    "chosen": "No. An association alone cannot establish causation.",
    "rejected": "Yes, any association establishes causation.",
}
assert all(isinstance(example[field], str) and example[field].strip()
           for field in ("prompt", "chosen", "rejected"))
assert example["chosen"] != example["rejected"]
print("Illustrative preference fields:", list(example))
print("Preferred response:", example["chosen"])
print("Rejected response:", example["rejected"])

Illustrative preference fields: ['prompt', 'chosen', 'rejected']
Preferred response: No. An association alone cannot establish causation.
Rejected response: Yes, any association establishes causation.


### Inspect public preference records when enabled

**Procedure.** Set `RUN_PUBLIC_DATA=1` to inspect the `reasoning_enhancement` source. With `HF_DATASETS_OFFLINE=1`, read a single existing Hugging Face Arrow cache; otherwise use `load_dataset`. Validate required fields, hash the full source content, and measure prompt and response reuse.

**Reading the result.** Pair counts describe automatically constructed preferences, not human judgments. Repeated prompts and candidate responses make row-level train/validation splits unsafe. The compact manifest records provenance and statistics; it contains no medical text or source rows.

In [2]:
# Inspect the full source schema, integrity, prompt groups, and provenance.
import hashlib
import json
import os
from collections import Counter, defaultdict
from pathlib import Path
from statistics import median

if os.getenv("RUN_PUBLIC_DATA", "0") == "1":
    from datasets import Dataset, config, load_dataset

    dataset_name = "HANI-LAB/Med-REFL-DPO"
    dataset_config = "reasoning_enhancement"
    source_split = "train"
    cache_file = None
    if os.getenv("HF_DATASETS_OFFLINE", "0") == "1":
        candidates = list(Path(config.HF_DATASETS_CACHE).glob(
            "HANI-LAB___med-refl-dpo/reasoning_enhancement/*/*/med-refl-dpo-train.arrow"
        ))
        if len(candidates) != 1:
            raise FileNotFoundError(f"Expected one Med-REFL-DPO Arrow cache; found {len(candidates)}")
        cache_file = candidates[0]
        source = Dataset.from_file(str(cache_file))
        print("Source mode: existing Hugging Face Arrow cache")
    else:
        source = load_dataset(dataset_name, dataset_config, split=source_split)
        print("Source mode: Hugging Face load_dataset")

    print("Dataset rows:", len(source))
    print("Source columns:", source.column_names)
    required = {"instruction", "input", "chosen", "rejected"}
    if not required.issubset(source.column_names):
        raise ValueError("Missing required preference fields; inspect the source schema.")

    rejected_records = Counter()
    prompt_counts = Counter()
    pair_counts = Counter()
    chosen_by_prompt = defaultdict(set)
    rejected_by_prompt = defaultdict(set)
    chosen_lengths = []
    rejected_lengths = []
    chosen_longer = 0
    empty_input_count = 0
    row_hashes = []
    first_valid = None
    for row in source:
        fields = [row.get(name) for name in ("instruction", "input", "chosen", "rejected")]
        if not all(isinstance(value, str) for value in fields):
            rejected_records["non_string_field"] += 1
            continue
        instruction, extra_input, chosen_raw, rejected_raw = fields
        canonical_row = json.dumps(fields, ensure_ascii=False, separators=(",", ":"))
        row_hashes.append(hashlib.sha256(canonical_row.encode("utf-8")).hexdigest())
        prompt = (instruction.strip() + "\n" + extra_input.strip()).strip()
        chosen = chosen_raw.strip()
        rejected = rejected_raw.strip()
        if not extra_input.strip():
            empty_input_count += 1
        if not prompt:
            rejected_records["empty_prompt"] += 1
        elif not chosen or not rejected:
            rejected_records["empty_response"] += 1
        elif chosen == rejected:
            rejected_records["identical_pair"] += 1
        else:
            prompt_counts[prompt] += 1
            pair_counts[(prompt, chosen, rejected)] += 1
            chosen_by_prompt[prompt].add(chosen)
            rejected_by_prompt[prompt].add(rejected)
            chosen_lengths.append(len(chosen))
            rejected_lengths.append(len(rejected))
            chosen_longer += len(chosen) > len(rejected)
            if first_valid is None:
                first_valid = {"prompt": prompt, "chosen": chosen, "rejected": rejected}

    valid_count = sum(prompt_counts.values())
    if valid_count == 0:
        raise ValueError("No valid preference pairs in the selected source.")
    if len(row_hashes) != len(source):
        raise ValueError("Non-string source fields prevent a complete content fingerprint.")
    fingerprint = hashlib.sha256("\n".join(sorted(row_hashes)).encode("ascii")).hexdigest()
    prompt_sizes = sorted(prompt_counts.values(), reverse=True)
    response_role_conflicts = sum(
        len(chosen_by_prompt[prompt] & rejected_by_prompt[prompt])
        for prompt in prompt_counts
    )
    duplicate_pairs = sum(count - 1 for count in pair_counts.values() if count > 1)
    print("Valid pairs:", valid_count, "Rejected records:", dict(rejected_records))
    print("Unique prompts:", len(prompt_counts), "Largest prompt group:", prompt_sizes[0])
    print("Repeated-prompt excess rows:", valid_count - len(prompt_counts))
    print("Exact duplicate pair excess:", duplicate_pairs,
          "Within-prompt response-role conflicts:", response_role_conflicts)
    print("Empty input fields:", empty_input_count)
    print("Median chosen/rejected characters:", median(chosen_lengths), median(rejected_lengths),
          "Chosen longer count:", chosen_longer)
    print("First prompt excerpt:", first_valid["prompt"][:160])
    print("First preferred response excerpt:", first_valid["chosen"][:160])
    print("First rejected response excerpt:", first_valid["rejected"][:160])
    if prompt_sizes[0] > 100 or valid_count - len(prompt_counts) > valid_count // 2:
        print("CAUTION: split by original prompt/case, not by pair rows; repeated cases dominate this source.")

    cache_sha256 = None
    if cache_file is not None:
        digest = hashlib.sha256()
        with cache_file.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
        cache_sha256 = digest.hexdigest()
    provenance = {
        "dataset": dataset_name, "config": dataset_config, "source_split": source_split,
        "source_revision": None, "source_content_fingerprint": fingerprint,
        "cache_arrow_sha256": cache_sha256, "source_rows": len(source),
        "valid_pairs": valid_count, "rejected_records": dict(rejected_records),
        "unique_prompts": len(prompt_counts), "largest_prompt_group": prompt_sizes[0],
        "duplicate_exact_pair_excess": duplicate_pairs,
        "within_prompt_response_role_conflicts": response_role_conflicts,
        "empty_input_fields": empty_input_count,
    }
    cwd = Path.cwd().resolve()
    if (cwd / "exploration" / "notebooks").is_dir():
        research = cwd / "exploration"
    elif cwd.name == "notebooks" and cwd.parent.name == "exploration":
        research = cwd.parent
    elif cwd.name == "exploration" and (cwd / "notebooks").is_dir():
        research = cwd
    else:
        raise FileNotFoundError("Open Jupyter from the repository root or exploration workspace.")
    manifest_path = research / "data" / "med_refl_reasoning_manifest.json"
    if manifest_path.exists():
        previous = json.loads(manifest_path.read_text(encoding="utf-8"))
        stable_keys = ("dataset", "config", "source_split", "source_content_fingerprint", "source_rows", "valid_pairs")
        if any(previous.get(key) != provenance[key] for key in stable_keys):
            raise RuntimeError("Preference source differs from the saved manifest; inspect before reuse.")
        if cache_sha256 is not None and previous.get("cache_arrow_sha256") not in (None, cache_sha256):
            raise RuntimeError("Cached Arrow bytes differ from the saved source.")
        print("Verified existing preference manifest:", manifest_path.resolve())
    else:
        manifest_path.write_text(json.dumps(provenance, indent=2) + "\n", encoding="utf-8")
        print("Saved preference manifest:", manifest_path.resolve())
else:
    print("Source inspection skipped. Set RUN_PUBLIC_DATA=1 to inspect original preferences.")

Source mode: existing Hugging Face Arrow cache
Dataset rows: 21326
Source columns: ['instruction', 'input', 'chosen', 'rejected']
Valid pairs: 21326 Rejected records: {}
Unique prompts: 355 Largest prompt group: 5400
Repeated-prompt excess rows: 20971
Exact duplicate pair excess: 0 Within-prompt response-role conflicts: 0
Empty input fields: 21326
Median chosen/rejected characters: 3151.0 3046.0 Chosen longer count: 11425
First prompt excerpt: A 61-year-old woman visits the clinic with a complaint of new-onset episodic abnormal body movements. She says her husband noticed it twice in the past week. Th
First preferred response excerpt: ## Thinking

I encountered a clinical challenge regarding a 61-year-old woman with new-onset episodic abnormal body movements. Let me think... she’s had two epi
First rejected response excerpt: ## Thinking

Now, I'm working through a clinical case where a 61-year-old woman who's experiencing new-onset episodic abnormal body movements. Let me think... T
CA

### Results

The cached Med-REFL-DPO reasoning source contained 21,326 pair rows but only 355 distinct exact prompts; the largest exact-prompt group had 5,400 rows. The `input` field was empty in the inspected source, and prompt text came from `instruction`.

### Interpretation

Prompt-level grouping is essential for selection and evaluation. Pair rows are not independent questions.

### Limitations

The rankings are automatically constructed, not independently verified human preferences. Exact-prompt checks do not eliminate semantic duplicates, style cues, or label ambiguity.

### Key Observations

Later DPO and reward-model experiments must disclose group-aware selection and preference-quality limits.


### Data quality questions
Which responses were generated automatically? Are preference scores independently verified? Could the rejected examples contain trivial stylistic cues that encourage shortcut learning? Document filtering and group-aware splitting where original case identifiers exist.

## Key observations and technical discussion
Inspecting a few pairs is not a data-quality audit. Report source, selected configuration, version, record count, and near-duplicate concerns before training.

## Q&A
**Question:** Why are automatically generated preference pairs not human judgments?

**Answer:** The labels result from a synthetic generation and filtering pipeline; their validity needs independent checks.

### Reproducibility note
Each printed example belongs to one of three categories: an illustrative unit check, a source-backed inspection, or an opt-in model experiment. Only real executed experiments with source/model identifiers and saved outputs can support research findings.